# Step 8: Final Open-Vocabulary Anomaly Evaluation (RbA)

This notebook implements the final stage of the project: evaluating the **Efficient Open-Vocabulary Multi-Task (EoMT)** model on anomaly segmentation benchmarks using the **Rejected by All (RbA)** methodology.

### Objectives:
1. **Reconstruct Dense Maps:** Map query-based mask predictions to dense spatial logit maps.
2. **Implement RbA Score:** Apply the $\tanh$-based scoring function to identify out-of-distribution (OOD) pixels.
3. **Temperature Scaling:** Optimize the scoring threshold for better calibration.
4. **Cross-Checkpoint Evaluation:** Generate the final results table for the project report.

## 1. Environment & Imports
We treat the `/eomt` directory as a library. Ensure the project root is in your Python path.

In [1]:
# @title
!pip install ood_metrics > /dev/null ## It will restart the session

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tifffile 2026.4.11 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which i

In [2]:
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null
!pip install gitignore_parser > /dev/null
!pip install lightning > /dev/null
!pip install -U "torchao>=0.16.1" > /dev/null

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 96.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [4]:
!pip install peft > /dev/null

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!ln -s /content/drive/MyDrive/FundGitHubProject/ /content/Fundamental_Project # symbolic shortcut

In [2]:
%cd Fundamental_Project/

/content/drive/MyDrive/FundGitHubProject


In [2]:
import os
import sys
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

# --- Proposed Fix for numpy incompatibility ---
# Upgrade numpy to version 2.0.0 (or a later compatible version) and reinstall dependent libraries
# that might have been compiled with a different numpy version.
# IMPORTANT: Allow these installations to complete without interruption.
!pip install -U numpy==2.0.0
!pip install -U torchvision timm

# Add project root and eomt directory to path
project_root = '/content/drive/MyDrive/FundGitHubProject/'
eomt_path = os.path.join(project_root, 'eomt')

# Add both directories to Python's module search path (sys.path)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if eomt_path not in sys.path:
    sys.path.insert(0, eomt_path)

print("Contents of project_root:", os.listdir(project_root) if os.path.exists(project_root) else "Not found")
print("Contents of eomt_path:", os.listdir(eomt_path) if os.path.exists(eomt_path) else "Not found")

from eomt.models.eomt import EoMT
from eval.iouEval import iouEval

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

  Using cached torchvision-0.27.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (5.5 kB)
  Using cached timm-1.0.27-py3-none-any.whl.metadata (40 kB)
  Using cached torch-2.12.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (31 kB)
  Using cached cuda_toolkit-13.0.2-py2.py3-none-any.whl.metadata (9.4 kB)
  Using cached nvidia_cublas-13.1.1.3-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached cuda_bindings-13.2.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (2.3 kB)
  Using cached nvidia_cudnn_cu13-9.20.0.48-py3-none-manylinux_2_27_x86_64.whl.metadata (1.9 kB)
  Using cached nvidia_cusparselt_cu13-0.8.1-py3-none-manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached nvidia_nccl_cu13-2.29.7-py3-none-manylinux_2_18_x86_64.whl.metadata (2.1 kB)
  Using cached nvidia_nvshmem_cu13-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (2.1 kB)
  Using cached triton-3.7.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.m

Contents of project_root: ['eval', '.git', 'LICENSE', 'coco-classes-mapping-master', 'docs', 'gitfunctions', 'eomt', 'trained_models', 'README_AI_Guide.md', '.ipynb_checkpoints', 'README.md', 'wandb', 'inference.ipynb', 'checkpoints', 'RbA-main', 'baselines', 'temperature_scaling-master', 'requirements.txt', 'results.txt', 'Step6.ipynb', 'Step7.ipynb', 'Step5.ipynb', 'Step4.ipynb', 'finetuning_results.json', 'Step5_Enhanced.ipynb', '.gitignore', 'FinalStep.ipynb']
Contents of eomt_path: ['.gitignore', 'LICENSE', '__init__.py', 'README.md', 'inference.ipynb', 'requirements.txt', 'main.py', 'configs', 'docs', 'training', 'datasets', 'models', 'data', '__pycache__', 'eomt_weights', '.ipynb_checkpoints', 'run_cli_silenced.py', 'wandb', 'README_AI_Guide.md', 'checkpoint_utils.py']


RuntimeError: operator torchvision::nms does not exist

## 2. The RbA Scoring Engine
Unlike standard models, EoMT uses query-based mask classification. We must bridge the gap by reconstructing a dense map before scoring.

**Formula:**
$$L_k(x) = \sum_{n=1}^{N} P_{nk} M_n(x)$$
$$RbA(x) = -\sum_{k=1}^{K} \tanh(L_k(x))$$

In [ ]:
def get_rba_anomaly_map(mask_cls, mask_pred, temperature=1.0):
    """
    Args:
        mask_cls: [B, N, C+1] raw class logits
        mask_pred: [B, N, H, W] raw mask logits
        temperature: Scaling factor for calibration
    """
    # 1. Apply Temperature and Softmax to get probabilities
    # Drop the last 'no-object' class index
    class_probs = F.softmax(mask_cls / temperature, dim=-1)[..., :-1] # Pay attention if the class are 19

    # 2. Sigmoid for mask probabilities
    mask_probs = torch.sigmoid(mask_pred)

    # 3. Dense Reconstruction via Einstein Summation
    # (Batch, Query, Class) x (Batch, Query, H, W) -> (Batch, Class, H, W)
    pixel_probs = torch.einsum('bnc,bnhw->bchw', class_probs, mask_probs)

    # 4. RbA Scoring (tanh variant)
    # Anomaly score is higher when all classes 'reject' the pixel
    anomaly_map = -torch.sum(torch.tanh(pixel_probs), dim=1)

    return anomaly_map

## 3. Model Loading from checkpoint
We load the weights into the EoMT architecture. Ensure you have the correct configuration parameters used during training.

In [3]:
# We can import directly from 'models' since the 'eomt' directory is in sys.path
# from models.eomt import EoMT # This import is redundant and may cause issues.
from eval.iouEval import iouEval


In [5]:
import sys
import os
import torch

# Ensure the 'eomt' directory is in sys.path to resolve internal 'models' imports
project_root = '/content/Fundamental_Project/' # or '/content/drive/MyDrive/FundGitHubProject/'
eomt_path = os.path.join(project_root, 'eomt')
if eomt_path not in sys.path:
    sys.path.insert(0, eomt_path)

from eomt.checkpoint_utils import get_finetuned_model

ckpt_path = '/content/Fundamental_Project/checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt'

eomt_ft = get_finetuned_model(ckpt_path)


W0526 15:48:11.992000 7757 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0526 15:48:12.023000 7757 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


--- Initializing Enhanced Architecture ---
Blocks: 3 | LoRA R: 8 | Backbone: vit_base_patch14_reg4_dinov2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights from: /content/Fundamental_Project/checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt
NOTE: Unexpected keys: 1
✅ Model ready for inference.


In [14]:
import sys
import os
import inspect
import torch
from torch.utils.data import DataLoader

# Ensure the project root is in sys.path so we can import from eval
if '/content/Fundamental_Project' not in sys.path:
    sys.path.append('/content/Fundamental_Project')

from eval.Validation_Dataset import anomaly_datasets

# Inspect the signature to see the exact arguments expected
sig = inspect.signature(anomaly_datasets.AnomalyDataModule.__init__)
print(f"AnomalyDataModule expects arguments: {sig}\n")

# -------------------------------------------------------------------
# Initialize the DataModules found in the script using 'root_dir'
# -------------------------------------------------------------------
DATA_DIR = '/content/Fundamental_Project/eval/Validation_Dataset'

dataloaders_to_evaluate = {}

# 1. Fishyscapes Static
try:
    fs_static = anomaly_datasets.FSStaticDM(root_dir=os.path.join(DATA_DIR, 'fs_static'))
    fs_static.setup()
    dataloaders_to_evaluate['FS Static'] = fs_static.val_dataloader()
except Exception as e:
    print(f"Failed to load FSStaticDM: {e}")

# 2. Fishyscapes Lost and Found
try:
    fs_laf = anomaly_datasets.FSLostFoundDM(root_dir=os.path.join(DATA_DIR, 'FS_LostFound_full'))
    fs_laf.setup()
    dataloaders_to_evaluate['FS Lost & Found'] = fs_laf.val_dataloader()
except Exception as e:
    print(f"Failed to load FSLostFoundDM: {e}")

# 3. Road Anomaly
try:
    road_anomaly = anomaly_datasets.RoadAnomalyDM(root_dir=os.path.join(DATA_DIR, 'RoadAnomaly'))
    road_anomaly.setup()
    dataloaders_to_evaluate['Road Anomaly'] = road_anomaly.val_dataloader()
except Exception as e:
    print(f"Failed to load RoadAnomalyDM: {e}")

print("\n--- Dataloader Setup Complete ---")
for name, loader in dataloaders_to_evaluate.items():
    print(f"{name}: {len(loader)} batches ready for evaluation.")

AnomalyDataModule expects arguments: (self, dataset_name, root_dir=None, batch_size=1, img_size=(640, 640))


--- Dataloader Setup Complete ---
FS Static: 30 batches ready for evaluation.
FS Lost & Found: 100 batches ready for evaluation.
Road Anomaly: 60 batches ready for evaluation.


## Evaluation

In [ ]:
def run_full_evaluation(model, dataloader, temp=1.0):
    """
    Runs inference and calculates AUPRC and FPR@95
    """
    from sklearn.metrics import average_precision_score
    from ood_metrics import fpr_at_95_tpr

    model.eval()
    all_anomaly_scores = []
    all_gt_masks = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating Anomaly Datasets"):
            images = batch['image'].to(device)
            gt = batch['label'] # Anomaly ground truth

            # Forward Pass
            mask_logits_list, class_logits_list = model(images)

            # Use final layer outputs
            m_pred = mask_logits_list[-1]
            c_cls = class_logits_list[-1]

            # Compute Score Map
            anomaly_map = get_rba_anomaly_map(c_cls, m_pred, temperature=temp)

            # Collect for metrics (ensure CPU and flattened)
            all_anomaly_scores.append(anomaly_map.cpu().numpy().flatten())
            all_gt_masks.append(gt.numpy().flatten())

    flat_scores = np.concatenate(all_anomaly_scores)
    flat_gt = np.concatenate(all_gt_masks)

    # Filter out ignore labels (usually 255)
    valid_mask = flat_gt != 255
    val_out = flat_scores[valid_mask]
    val_label = flat_gt[valid_mask]

    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)

    return {'auprc': prc_auc * 100.0, 'fpr95': fpr * 100.0}

## 5. Final Results Table
Execute the evaluation across the three main checkpoints to complete your report.

In [ ]:
import gc
import torch
import numpy as np
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import average_precision_score
from ood_metrics import fpr_at_95_tpr
import importlib
import eomt.checkpoint_utils

# --- RESTORE checkpoint_utils.py to original state ---
file_path = '/content/Fundamental_Project/eomt/checkpoint_utils.py'
with open(file_path, 'r') as f:
    src = f.read()

if "num_q=200" in src:
    src = src.replace("def get_finetuned_model(checkpoint_path, device='cuda', num_q=200):", "def get_finetuned_model(checkpoint_path, device='cuda'):")
    src = src.replace("'num_q': num_q,", "'num_q': 200,")
    with open(file_path, 'w') as f:
        f.write(src)

importlib.reload(eomt.checkpoint_utils)
from eomt.checkpoint_utils import get_finetuned_model
from eomt.models.vit import ViT
from eomt.models.eomt import EoMT

# Define device directly in this cell to prevent NameErrors
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Loader for the baseline model (100 queries, no LoRA)
def get_base_model(checkpoint_path, device='cuda'):
    print("\n--- Initializing Base Architecture ---")
    encoder = ViT(img_size=(640, 640), backbone_name='vit_base_patch14_reg4_dinov2')
    model = EoMT(
        encoder=encoder,
        num_classes=19,
        num_q=100,
        num_blocks=3,
        masked_attn_enabled=True
    )

    print(f"Loading weights from: {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    state_dict = checkpoint.get('state_dict', checkpoint)

    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('network.'): k = k[len('network.'):]
        elif k.startswith('model.'): k = k[len('model.'):]
        new_state_dict[k] = v

    model.load_state_dict(new_state_dict, strict=False)
    model.to(device)
    model.eval()
    print("\u2705 Base Model ready for inference.")
    return model

# Ensure get_rba_anomaly_map is defined in the current scope
def get_rba_anomaly_map(mask_cls, mask_pred, temperature=1.0):
    class_probs = F.softmax(mask_cls / temperature, dim=-1)[..., :-1]
    mask_probs = torch.sigmoid(mask_pred)
    pixel_probs = torch.einsum('bnc,bnhw->bchw', class_probs, mask_probs)
    anomaly_map = -torch.sum(torch.tanh(pixel_probs), dim=1)
    return anomaly_map

# Ensure run_full_evaluation is defined in the current scope
def run_full_evaluation(model, dataloader, temp=1.0):
    model.eval()
    all_anomaly_scores = []
    all_gt_masks = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating Anomaly Datasets"):
            images = batch['image'].to(device)
            gt = batch['label'] # Anomaly ground truth

            # Forward Pass
            mask_logits_list, class_logits_list = model(images)

            # Use final layer outputs
            m_pred = mask_logits_list[-1]
            c_cls = class_logits_list[-1]

            # Compute Score Map
            anomaly_map = get_rba_anomaly_map(c_cls, m_pred, temperature=temp)

            # Upsample anomaly map to match ground truth resolution
            anomaly_map = F.interpolate(
                anomaly_map.unsqueeze(1),
                size=gt.shape[-2:],
                mode='bilinear',
                align_corners=False
            ).squeeze(1)

            # Collect for metrics (ensure CPU and flattened)
            all_anomaly_scores.append(anomaly_map.cpu().numpy().flatten())
            all_gt_masks.append(gt.numpy().flatten())

    flat_scores = np.concatenate(all_anomaly_scores)
    flat_gt = np.concatenate(all_gt_masks)

    # Filter out ignore labels (usually 255)
    valid_mask = flat_gt != 255
    val_out = flat_scores[valid_mask]
    val_label = flat_gt[valid_mask]

    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)

    return {'auprc': prc_auc * 100.0, 'fpr95': fpr * 100.0}

checkpoints = [
    "/content/Fundamental_Project/eomt/eomt_weights/eomt_cityscapes_lightning.ckpt",
    "/content/Fundamental_Project/checkpoints/cityscapes_enhanced/eomt-enhanced-epoch=23-val_iou_all=0.00.ckpt"
]

print("| Checkpoint | Dataset | AUPRC | FPR@95 |")
print("|------------|---------|-------|--------|")

for ckpt in checkpoints:
    # Load the correct model architecture based on the checkpoint name
    if 'lightning' in ckpt:
        model = get_base_model(ckpt, device)
    else:
        model = get_finetuned_model(ckpt)
        model = model.to(device)
        model.eval()

    for ds_name, ds_loader in dataloaders_to_evaluate.items():
        res = run_full_evaluation(model, ds_loader)
        ckpt_name = ckpt.split('/')[-1]
        print(f"| {ckpt_name} | {ds_name} | {res['auprc']:.2f} | {res['fpr95']:.2f} |")

    # Cleanup memory before loading the next checkpoint
    del model
    torch.cuda.empty_cache()
    gc.collect()

| Checkpoint | Dataset | AUPRC | FPR@95 |
|------------|---------|-------|--------|

--- Initializing Base Architecture ---


In [17]:
import inspect
from eomt.checkpoint_utils import get_finetuned_model

print("Signature of get_finetuned_model:", inspect.signature(get_finetuned_model))

# Let's also print its source code to see how it initializes the model
print("\n--- Source Code ---")
print(inspect.getsource(get_finetuned_model))

Signature of get_finetuned_model: (checkpoint_path, device='cuda')

--- Source Code ---
def get_finetuned_model(checkpoint_path, device='cuda'):
    """
    Loads the fine-tuned EoMT model using the 'Step 5 Enhanced' configuration.
    
    Args:
        checkpoint_path (str): Path to the .ckpt or .pth file.
        device (str): 'cuda' or 'cpu'.
        
    Returns:
        torch.nn.Module: The loaded model in eval mode.
    """
    
    # 1. Configuration (Matching Step 5 Enhanced)
    # These MUST match the parameters used during the successful 76% mIoU run.
    CONFIG = {
        'img_size': (640, 640),
        'num_classes': 19,
        'num_q': 200,
        'num_blocks': 3,
        'lora_r': 8,
        'lora_alpha': 16,
        'backbone': 'vit_base_patch14_reg4_dinov2'
    }

    print(f"--- Initializing Enhanced Architecture ---")
    print(f"Blocks: {CONFIG['num_blocks']} | LoRA R: {CONFIG['lora_r']} | Backbone: {CONFIG['backbone']}")

    # 2. Initialize Base Vision Transfor